# 03 — MQTT Ingestion (paho-mqtt + SDK verify)

| Layer | Format | Example |
|---|---|---|
| MQTT publish | `{org}/{project}/{collection}` | `myorg/myproject/sensors` |
| Kafka | `{org}.{project}.{collection}` | `myorg.myproject.sensors` |
| Cassandra | ks `{org}` tbl `{project}_{collection}` | — |

Publish with QoS 1 as `project_id`/`api_key`, verify via SDK `data.get`.

In [ ]:
%pip install -q "nostradamus-ioto-sdk @ git+https://github.com/robotics-4-all/nostradamus-ioto-sdk.git@4f8e7fc7c28c859eefed9f7382d1b148a5c6eeb8" "paho-mqtt==2.1.0" python-dotenv

In [ ]:
import json, os, random, time
from datetime import datetime, timezone
from dotenv import load_dotenv
load_dotenv()
BASE_URL = os.getenv("CENOTOO_BASE_URL", "http://localhost:8000")
USERNAME = os.getenv("CENOTOO_USERNAME", "")
PASSWORD = os.getenv("CENOTOO_PASSWORD", "")
MQTT_HOST = os.getenv("MQTT_HOST", "localhost")
MQTT_PORT = int(os.getenv("MQTT_PORT", "1883"))
assert BASE_URL and USERNAME and PASSWORD, "SKIP: set CENOTOO_BASE_URL/USERNAME/PASSWORD"

def unique_name(p="nb"):
    return f"{p}_{time.strftime('%Y%m%d%H%M%S')}_{random.randint(1000,9999)}"

def wait_for(fn, pred, timeout=60, interval=2, desc="c"):
    import time as _t
    end = _t.monotonic() + timeout
    while _t.monotonic() < end:
        v = fn()
        try:
            if pred(v):
                return v
        except Exception:
            pass
        _t.sleep(interval)
    raise TimeoutError(desc)

In [ ]:
from nostradamus_ioto_sdk import NostradamusClient
from nostradamus_ioto_sdk.models import KeyType
admin = NostradamusClient(base_url=BASE_URL, username=USERNAME, password=PASSWORD)
# NOTE: for MQTT ACL the topic org/project must match Postgres names for this project.
# Use display names you own; here we create fresh project/collection and use THEIR names.
project = admin.projects.create(name=unique_name("nbmqtt"), description="nb03", tags=["notebook", "03"])
collection = admin.collections.create(project_id=project.project_id, name=unique_name("sensors"),
    description="nb03", tags=["notebook", "03"],
    collection_schema={"key": "K", "timestamp": "2026-01-01T00:00:00Z", "temperature": 20.0})
wk = admin.project_keys.create(project.project_id, KeyType.WRITE)
ORG = admin.projects.get(project.project_id).name  # display org segment: see README for prod mapping
print("project:", project.name, "collection:", collection.name)

**Prod ACL rule:** `username = <project_id>` (UUID), `password = <write|master api_key>`; topic must be `{org}/{project}/{collection}` matching Postgres. Subscribe needs the `cenotoo-bridge` superuser — we verify via SDK/REST instead.

In [ ]:
import paho.mqtt.client as mqtt
TOPIC = f"{project.name}/{project.name}/{collection.name}"  # replace left segments with your ORG when names differ
print("publishing to:", TOPIC, f"({MQTT_HOST}:{MQTT_PORT})")
probe = unique_name("mqttprobe")
payload = {"key": probe, "timestamp": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
           "temperature": 22.5, "humidity": 58}
c = mqtt.Client(callback_api_version=mqtt.CallbackAPIVersion.VERSION2, client_id=f"nb-mqtt-{random.randint(0,9999)}")
c.username_pw_set(str(project.project_id), wk.api_key)
c.connect(MQTT_HOST, MQTT_PORT, keepalive=60)
c.loop_start()
time.sleep(1)
r = c.publish(TOPIC, json.dumps(payload), qos=1)
r.wait_for_publish(timeout=5)
print("published:", r.is_published())
time.sleep(0.5)
c.loop_stop(); c.disconnect()

In [ ]:
# Negative: wrong topic is ACL-dropped (publish succeeds at MQTT level, nothing lands)
print("negative demo: publish to wrong/project/topic is dropped by bridge (3-segment rule / ACL)")

In [ ]:
found = wait_for(lambda: admin.data.get(project_id=project.project_id, collection_id=collection.collection_id,
    filters=[{"property_name": "key", "operator": "eq", "property_value": probe}], limit=10),
    lambda resp: len(resp.data) >= 1, desc="mqtt record readable via SDK")
print("verify OK: mqtt -> kafka -> cassandra -> SDK read")

In [ ]:
import contextlib
with contextlib.suppress(Exception):
    admin.collections.delete(project.project_id, collection.collection_id)
with contextlib.suppress(Exception):
    admin.projects.delete(project.project_id)
admin.close()
print("cleanup done")